# 04 · CatBoost versus Gradient Boosting

**Question:** Can CatBoost outperform the seven-feature scikit-learn Gradient Boosting baseline?

The two models are evaluated on exactly the same **5 folds × 3 repeats (15 validation scores)**. CatBoost receives categorical columns as strings and handles the remaining numeric missing values; scikit-learn Gradient Boosting uses imputation and one-hot encoding fitted within each fold.

**Original experiment:** `10_catboost.ipynb`.

## 1. Load data

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
from pathlib import Path

# Works when running Jupyter from either the repository root or notebooks/.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_dir = next(
    (folder for folder in candidates if (folder / 'data' / 'train.csv').exists()),
    None,
)
if project_dir is None:
    raise FileNotFoundError(
        'data/train.csv not found. Download train.csv and test.csv from '
        'https://www.kaggle.com/competitions/titanic/data and put them in data/.'
    )

In [ ]:
train = pd.read_csv(project_dir / 'data' / 'train.csv')
test = pd.read_csv(project_dir / 'data' / 'test.csv')

features = ['Pclass', 'Sex', 'Age', 'Fare', 'SibSp', 'Parch', 'Embarked']
categorical = ['Pclass', 'Sex', 'Embarked']
X = train[features].copy()
y = train['Survived'].copy()
X_test = test[features].copy()

## 2. Build the two comparable models

In [ ]:
def prepare_catboost(df):
    result = df.copy()
    for column in categorical:
        result[column] = result[column].fillna('Unknown').astype(str)
    return result


def make_catboost():
    return CatBoostClassifier(
        iterations=400,
        depth=4,
        learning_rate=0.04,
        loss_function='Logloss',
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=2,
    )


def make_gradient_boosting():
    preprocessor = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'),
         ['Age', 'Fare', 'SibSp', 'Parch']),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical),
    ], sparse_threshold=0.0)
    return Pipeline([
        ('preprocessing', preprocessor),
        ('classifier', GradientBoostingClassifier(random_state=42)),
    ])


X_cat = prepare_catboost(X)
X_test_cat = prepare_catboost(X_test)

## 3. Repeated, paired model comparison

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
splits = list(cv.split(X, y))
gb_scores, cat_scores = [], []

for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    gb = make_gradient_boosting()
    gb.fit(X.iloc[train_idx], y.iloc[train_idx])
    gb_predictions = gb.predict(X.iloc[val_idx])
    gb_scores.append(accuracy_score(y.iloc[val_idx], gb_predictions))

    cat = make_catboost()
    cat.fit(X_cat.iloc[train_idx], y.iloc[train_idx],
            cat_features=categorical)
    cat_predictions = cat.predict(X_cat.iloc[val_idx]).astype(int).ravel()
    cat_scores.append(accuracy_score(y.iloc[val_idx], cat_predictions))

    print(f'Fold {fold:02d}: GB={gb_scores[-1]:.4f}  CatBoost={cat_scores[-1]:.4f}')

gb_scores = np.asarray(gb_scores)
cat_scores = np.asarray(cat_scores)
results = pd.DataFrame([
    {'Model': 'Gradient Boosting', 'Mean CV': gb_scores.mean(),
     'Std': gb_scores.std()},
    {'Model': 'CatBoost', 'Mean CV': cat_scores.mean(),
     'Std': cat_scores.std()},
]).sort_values('Mean CV', ascending=False)
display(results.round(4))

difference = cat_scores - gb_scores
print(f'CatBoost – Gradient Boosting mean difference: {difference.mean():+.4f}')
print('Better / worse / tied folds:',
      int((difference > 0).sum()),
      int((difference < 0).sum()),
      int((difference == 0).sum()))

### Outcome

**Recorded original repeated CV:** Gradient Boosting **0.8294 ± 0.0177**; CatBoost **0.8230 ± 0.0214**. CatBoost was better in 6 folds, worse in 8, and tied in 1. The mean paired accuracy difference (CatBoost − Gradient Boosting) was **−0.0064**.

The original CatBoost Kaggle public leaderboard score was **0.77511**, again below the simpler Gradient Boosting submission (**0.79665**). This does not show CatBoost is generally worse; it shows that **this configuration did not improve this experiment**.

## 4. Reproduce the CatBoost submission (optional)

In [ ]:
final_model = make_catboost()
final_model.fit(X_cat, y, cat_features=categorical)
predictions = final_model.predict(X_test_cat).astype(int).ravel()

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions,
})
assert len(submission) == len(test)
assert submission['PassengerId'].is_unique
assert submission['Survived'].isin([0, 1]).all()
assert not submission.isna().any().any()

submission_dir = project_dir / 'submissions'
submission_dir.mkdir(exist_ok=True)
path = submission_dir / 'submission_catboost_v6.csv'
submission.to_csv(path, index=False)
print(f'Saved {path.relative_to(project_dir)}')
display(submission.head())